# 배경 생성 PoC — sd-turbo
그라데이션(임시 배경)을 AI 생성 배경으로 교체하기 위한 실험.
프롬프트 → 배경 생성 → 누끼·문구와 합성까지 확인한다.

In [ ]:
import torch
from diffusers import AutoPipelineForText2Image

pipe = AutoPipelineForText2Image.from_pretrained(
    "stabilityai/sd-turbo", torch_dtype=torch.float16
).to("cuda")

In [ ]:
prompt = "warm cozy cafe interior, wooden table, soft morning light, blurred background, product photo backdrop"
bg = pipe(prompt=prompt, num_inference_steps=2, guidance_scale=0.0).images[0]
bg

In [ ]:
from PIL import Image
from app_core.background import remove_background
from app_core.compose import compose_ad

cut = remove_background(Image.open("테스트사진.jpg"))
ad = compose_ad(cut, "크로플 출시 기념!", "지금 바로 3,500원에 만나요!", background=bg)
ad.save("첫_AI배경_광고.png")

small = ad.copy()
small.thumbnail((420, 420))
small

In [ ]:
import importlib
import app_core.compose
importlib.reload(app_core.compose)
from app_core.compose import compose_ad

ad = compose_ad(cut, "크로플 출시 기념!", "지금 바로 3,500원에 만나요!", background=bg)
small = ad.copy()
small.thumbnail((420, 420))
small

In [ ]:
old = Image.open("첫_AI배경_광고.png")   # 그림자 없던 아까 버전
both = Image.new("RGB", (old.width * 2 + 20, old.height), (30, 30, 30))
both.paste(old, (0, 0))
both.paste(ad, (old.width + 20, 0))
sm = both.copy()
sm.thumbnail((900, 450))
sm

In [ ]:
from PIL import Image
from app_core.background import remove_background
from app_core.compose import compose_ad
from app_core.gen_background import generate_background

bg = generate_background("warm cozy cafe interior, wooden table, soft morning light, blurred background")
cut = remove_background(Image.open("테스트사진.jpg"))
ad = compose_ad(cut, "크로플 출시 기념!", "지금 바로 3,500원에 만나요!", background=bg)
small = ad.copy()
small.thumbnail((420, 420))
small

In [ ]:
from dotenv import load_dotenv

load_dotenv()  # .env 파일의 OPENAI_API_KEY를 환경변수로 올림

from app_core.prompt_builder import build_bg_prompt

p = build_bg_prompt("분식집", "신규 오픈", "활기찬 점심")
print(p)

In [ ]:
bg2 = generate_background(p)
cut2 = remove_background(Image.open("테스트사진.jpg"))
ad2 = compose_ad(cut2, "코드잇분식 오픈!", "8월 31일, 직장인 점심 특가", background=bg2)
small = ad2.copy()
small.thumbnail((420, 420))
small

In [ ]:
from app_core.photo_store import load_photo, save_photo

num = save_photo(Image.open("테스트사진.jpg"))
print(f"보관 완료 — 번호표 {num}번")

again = load_photo(num)
again.size

In [ ]:
import importlib

import app_core.pipeline

importlib.reload(app_core.pipeline)
from app_core.pipeline import generate_ad

ad = generate_ad(
    "분식집", "코드잇분식 오픈!", "8월 31일 점심 특가", situation="신규 오픈", tone="따뜻한", photo_id=1
)
small = ad.copy()
small.thumbnail((420, 420))
small

## 실험 기록 (2026-08-10)
- sd-turbo 2스텝, 영어 프롬프트 → 카페 배경 성공 (한글 프롬프트는 엉뚱한 그림 — 번역 부품 필요)
- AI 배경 + 누끼 + 두 줄 문구 첫 합성 성공 (첫_AI배경_광고.png)
- 개선 2호 후보: 사진 배경에서 글자 가독성 (반투명 띠 / 흰 글자+그림자)
- 개선 3호 후보: 제품 아래 그림자 (스티커 느낌 제거)
- 번역 부품: "no food" 부정어는 역효과(음식 소환) → 긍정형("empty clean surface")으로 전환
- photo_store(5호)·pipeline(6호) 검증 — generate_ad() 한 번으로 완성 광고

In [ ]:
import torch


def try_bg(prompt):
    g = torch.Generator("cuda").manual_seed(42)
    img = pipe(prompt=prompt, num_inference_steps=2, guidance_scale=0.0, generator=g).images[0]
    small = img.copy()
    small.thumbnail((300, 300))
    return small


try_bg("close-up of an empty wooden tabletop, blurred snack bar interior in the background, soft warm light")

In [ ]:
try_bg("bright lively snack bar interior, close-up of an empty clean surface in the foreground, soft light")

In [ ]:
try_bg("close-up of an empty wooden tabletop, blurred snack bar interior in the background, soft warm light, low camera angle, shallow depth of field")

In [ ]:
import importlib

import app_core.pipeline

importlib.reload(app_core.pipeline)
from app_core.pipeline import generate_ad
from app_core.schema import AdBrief, CopyCandidate, Store

store = Store(id=1, user_id=1, industry="cafe", name="코드잇분식", address="서울시 마포구 연남동 1-2")
brief = AdBrief(
    goal="image", product="오므라이스", price=9900,
    situation="신규 오픈", tone="따뜻한", photo_id=1,
)
copy = CopyCandidate(headline="코드잇분식 오픈!", sub="8월 31일, 오므라이스 9,900원")

ad = generate_ad(brief, store, copy)
small = ad.copy()
small.thumbnail((420, 420))
small

In [ ]:
import importlib

import app_core.compose
import app_core.pipeline

importlib.reload(app_core.compose)
importlib.reload(app_core.pipeline)
from app_core.pipeline import generate_ad
from app_core.schema import AdBrief, CopyCandidate, Store

store = Store(id=1, user_id=1, industry="cafe", name="코드잇스터디카페", address="서울시 마포구 연남동 1-2")
brief = AdBrief(
    goal="image", product="스터디카페 이용권", price=0,
    situation="신규 오픈", tone="차분하고 집중되는", photo_id=None,
)
copy = CopyCandidate(headline="8월 31일 오픈!", sub="조용한 나만의 공간")

ad = generate_ad(brief, store, copy)
small = ad.copy()
small.thumbnail((420, 420))
small